# AI for the problem and the hypothesis {#sec-ai-conceptualise}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/book-ai/notebooks/c18-ai-conceptualise.ipynb)

@sec-story-to-diagram turned four interviews and a data series into the products of the first stage: variables, a causal loop diagram, a reference mode, a boundary and a dynamic hypothesis. All of it was done by hand. This chapter does the same work again with language models beside us, at each step, and checks every output by code before it is allowed near the model. The aim is not speed for its own sake. It is a second reader who never tires, whose mistakes we know how to catch.

```
 interviews -------> [ generative ] ----> variables, links ------+
                                                                 |
 problem statement   [ reasoning  ] ----> critique, claims ------+
 and reference mode                                              |
                                                                 v
 checked diagram --> [   agent    ] ----> rival hypotheses --> checks by code:
 and the data         (tools run                               quotes found,
                       in Python)                              units and kinds,
                                                               claims recomputed,
 personas ---------> rehearsal only ----> questions for        numbers from tools
                                          real people                |
                                                                     v
                                               the modeller decides, and records it
```

*The first stage with AI partners. Three kinds of model draft, critique and act; code checks each output; a person decides.*

The stage is the one the field has long called problem articulation and dynamic hypothesis [@richardson1981; @sterman2000; @martinezmoyano2013]. Recent work shows that language models can draft causal loop diagrams from text [@hosseinichimeh2024], and also how they mislead the unwary [@akhavan2024]. We use three kinds of AI on purpose, as @sec-ai-partners introduced. A **generative** model drafts, a **reasoning** model critiques, and an **agent** tests hypotheses with tools we write.

**Who it is for.** Modellers who want to use language models in the first stage of a project without letting them into the evidence unchecked.

**Who can skip what.** If you already check model drafts by code, read section 4 (the agent) and the failure gallery. Sections 1 and 3 repeat chapter 3's work and show what the model adds and gets wrong.

**Prerequisites.** @sec-story-to-diagram, whose notes, tables, loop finder and reference mode we reuse, and @sec-ai-partners, which explains replies, replay files and the contract. Python dictionaries, pandas and regular expressions at an introductory level.

**Time.** About three hours, including the exercises.

In [ ]:
# Setup: run this cell first. In Colab it installs loopbricks (ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@book-ai#subdirectory=python"])
    import loopbricks as lb
import os, re, pandas as pd
from loopbricks import ai
# Use local copies of the data and the recorded model replies if there are
# any, otherwise read them from GitHub.
GITHUB = ("https://raw.githubusercontent.com/waelrash1/loopbricks/main/"
          "notebooks/")
DATA = "data/waiting_list.csv" if os.path.exists("data/waiting_list.csv") \
    else GITHUB + "data/waiting_list.csv"
REPLAY = "data/ai/c18.json" if os.path.exists("data/ai/c18.json") \
    else GITHUB + "data/ai/c18.json"
wl = pd.read_csv(DATA, parse_dates=["month"])
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cycler

SEED = 4                               # one seed per chapter
rng = np.random.default_rng(SEED)      # (this chapter draws no random numbers)
# Colour-blind-safe colours (Okabe and Ito) with a different line style
# for each series, so every figure also reads in greyscale.
OKABE_ITO = ["#000000", "#E69F00", "#0072B2", "#009E73", "#D55E00",
             "#CC79A7", "#56B4E9"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1, 1, 1)),
          (0, (1, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "figure.dpi": 100})
print("model:", ai.MODEL, "| replies from:", REPLAY)

## Learning outcomes

By the end of this chapter you will be able to:

1. **Draft** a variable list from interviews with a language model, and **check** its quotes, units and kinds by code (Try it 1).
2. **Critique** a problem statement with a reasoning model, and **test** each factual claim in the critique against the data (Try it 2).
3. **Design** a prompt that asks for causal links with evidence, and **diagnose** a draft by its quotes, polarities and loops (Try it 3).
4. **Run** an agent that tests rival dynamic hypotheses with Python tools, and **verify** that every number in its answer came from a tool (Try it 4).
5. **Rehearse** an interview with synthetic personas, and **keep** what they say out of the evidence (Try it 5).
6. **Recognise** the ways AI fails at this stage, and **catch** each one with a check (Try it 6).

### How the replies in this chapter were made

Every model call goes through `loopbricks.ai`, with `replay=REPLAY`. The replies are recorded in `data/ai/c18.json`, keyed by a hash of the whole request, so the chapter runs offline, free and the same every time. **These replies were written for the book, with claude-opus-5-5 as the model they imitate, not returned by a live call.** Each record says so in its `how` field. Some contain faults on purpose, for teaching, and the text says where.

To run the chapter against a live model, `pip install anthropic`, set `ANTHROPIC_API_KEY` and `LB_AI_LIVE=1`, and change a prompt (or delete the replay file). A request with no recorded reply then goes to the model, and its reply is recorded with `how` set to a live call. Without those settings a changed request raises `ai.MissingReply`. Live replies will differ from ours, and the checks are written so that they still catch the same kinds of fault.

The cell below repeats chapter 3's products so that this chapter runs on its own. They are the four notes, the variable table, the edge list, `find_loops`, the loop names and `name_loops`.

In [ ]:
# Chapter 3's products, repeated so that this chapter runs on its own.
notes = [
    {"who": "Consultant", "said":
     "We run every clinic at full stretch. When the list goes up, management "
     "asks for extra Saturday clinics and we do them, but you can only do "
     "that for so long. Two of our registrars left last year, burnt out, "
     "and their posts took nine months to fill. And patients arrive further "
     "along than they used to: a knee we could have managed with physio "
     "two years ago now needs surgery and three follow-up visits."},
    {"who": "Manager", "said":
     "The board watches one number, the patients waiting over a year. When "
     "it climbs, we buy appointments from private providers, and it comes "
     "down. But that money comes out of the same budget, so next year we "
     "have less to hire our own staff. A validation exercise takes a few "
     "thousand off the list, and then it creeps back."},
    {"who": "GP", "said":
     "When the orthopaedic wait was six months, I referred anyone who might "
     "benefit. Now I tell people it is two years, and many say there is no "
     "point, or pay to go private if they can. The referrals you see are "
     "not the need that is out there."},
    {"who": "Patient advocate", "said":
     "Being on the list is not neutral. People get worse while they wait, "
     "they lose work, they come to us in pain. Some are taken off the list "
     "by a letter they never saw, and the hospital counts it as progress. "
     "And the population gets older every year, so need only goes one way."},
]
variables = pd.DataFrame([
    ("Need in the population", "auxiliary", "patients/month",
     "Patient advocate", "no"),
    ("Referrals", "flow", "patients/month", "GP", "no"),
    ("Waiting list", "stock", "patients", "all four", "yes"),
    ("Waiting time", "auxiliary", "months", "GP", "partly"),
    ("Leaving the list", "flow", "patients/month",
     "GP, Manager, Patient advocate", "no"),
    ("Severity", "stock", "index (2014 = 1)",
     "Consultant, Patient advocate", "no"),
    ("Appointments per patient", "auxiliary", "appointments/patient",
     "Consultant", "no"),
    ("Treatments", "flow", "patients/month", "Consultant", "no"),
    ("Capacity", "auxiliary", "appointments/month", "Consultant", "no"),
    ("Pressure", "auxiliary", "dimensionless", "Manager, Consultant", "no"),
    ("Overtime", "auxiliary", "extra clinics/month", "Consultant", "no"),
    ("Fatigue", "stock", "index (rested = 0)", "Consultant", "no"),
    ("Staff leaving", "flow", "clinicians/month", "Consultant", "no"),
    ("Staff", "stock", "clinicians", "Consultant, Manager", "no"),
    ("Outsourcing", "flow", "patients/month", "Manager", "no"),
    ("Budget for own staff", "auxiliary", "euros/year", "Manager", "no"),
], columns=["variable", "kind", "units", "source", "measured"])
edges = pd.DataFrame([
    ("Need in the population", "Referrals", "+", False, "Patient advocate"),
    ("Referrals", "Waiting list", "+", False, "inflow"),
    ("Treatments", "Waiting list", "-", False, "outflow"),
    ("Waiting list", "Waiting time", "+", False, "definition"),
    ("Waiting time", "Referrals", "-", False, "GP"),
    ("Waiting time", "Leaving the list", "+", False, "GP"),
    ("Leaving the list", "Waiting list", "-", False, "outflow"),
    ("Waiting time", "Severity", "+", True, "Patient advocate"),
    ("Severity", "Appointments per patient", "+", False, "Consultant"),
    ("Appointments per patient", "Treatments", "-", False, "Consultant"),
    ("Capacity", "Treatments", "+", False, "Consultant"),
    ("Waiting list", "Pressure", "+", False, "Manager"),
    ("Pressure", "Overtime", "+", False, "Consultant"),
    ("Overtime", "Capacity", "+", False, "Consultant"),
    ("Overtime", "Fatigue", "+", True, "Consultant"),
    ("Fatigue", "Staff leaving", "+", False, "Consultant"),
    ("Staff leaving", "Staff", "-", False, "outflow"),
    ("Staff", "Capacity", "+", False, "Consultant"),
    ("Pressure", "Outsourcing", "+", False, "Manager"),
    ("Outsourcing", "Treatments", "+", False, "Manager"),
    ("Outsourcing", "Budget for own staff", "-", True, "Manager"),
    ("Budget for own staff", "Staff", "+", True, "Manager"),
], columns=["cause", "effect", "sign", "delay", "source"])


def find_loops(edges):
    """Every simple feedback loop in an edge list, as (path, "R" or "B")."""
    out = {}
    for cause, effect, sign in edges[["cause", "effect", "sign"]].itertuples(
            index=False):
        out.setdefault(cause, []).append((effect, sign))
    names = sorted(set(edges["cause"]) | set(edges["effect"]))
    loops = []

    def follow(start, here, path, signs):
        for nxt, sign in out.get(here, []):
            if nxt == start:
                negatives = (signs + [sign]).count("-")
                loops.append((path, "R" if negatives % 2 == 0 else "B"))
            elif nxt > start and nxt not in path:
                follow(start, nxt, path + [nxt], signs + [sign])

    for start in names:
        follow(start, start, [start], [])
    turned = []                        # start each loop at the waiting list
    for path, polarity in loops:
        k = path.index("Waiting list") if "Waiting list" in path else 0
        turned.append((path[k:] + path[:k], polarity))
    return turned


LOOP_NAMES = {
    frozenset({"Waiting list", "Waiting time", "Referrals"}):
        ("B1", "Discouraged referrals"),
    frozenset({"Waiting list", "Waiting time", "Leaving the list"}):
        ("B2", "Giving up"),
    frozenset({"Waiting list", "Pressure", "Overtime", "Capacity",
               "Treatments"}):
        ("B3", "Working harder"),
    frozenset({"Waiting list", "Pressure", "Outsourcing", "Treatments"}):
        ("B4", "Buying activity"),
    frozenset({"Waiting list", "Waiting time", "Severity",
               "Appointments per patient", "Treatments"}):
        ("R1", "Sicker by the time they are seen"),
    frozenset({"Waiting list", "Pressure", "Overtime", "Fatigue",
               "Staff leaving", "Staff", "Capacity", "Treatments"}):
        ("R2", "Burnout"),
    frozenset({"Waiting list", "Pressure", "Outsourcing",
               "Budget for own staff", "Staff", "Capacity", "Treatments"}):
        ("R3", "Borrowing from next year"),
}


def name_loops(loops):
    """A table of loops with their names; a name must match its polarity."""
    rows = []
    for path, polarity in loops:
        loop_id, name = LOOP_NAMES.get(frozenset(path), ("?", "unnamed"))
        assert loop_id == "?" or loop_id[0] == polarity, f"{name}: polarity"
        rows.append((loop_id, name, polarity, len(path), " -> ".join(path)))
    table = pd.DataFrame(rows, columns=["id", "name", "polarity", "links",
                                        "path"])
    return table.sort_values("id").reset_index(drop=True)


total = wl.set_index("month")["total"]
share = 100 * wl.set_index("month")["w12_plus"] / total
print(f"{len(notes)} notes, {len(variables)} variables, {len(edges)} links, "
      f"{len(find_loops(edges))} loops")

Four notes, 16 variables, 22 links and 7 loops, as chapter 3 left them. One more piece of setup. The contract of @sec-ai-partners asks us to record what was asked of every AI output: prompt, model, date and replay key. The cell below wraps `ai.request_key` so that each key the chapter uses is noted, and `note` adds a row to a record we complete in the running case. `norm` makes text lower case with single spaces, so a quote check ignores line breaks and capitals.

In [ ]:
# Every request has a replay key (a hash of everything sent). This small
# logger notes each key the chapter uses, for the record at the end.
USED = []
_request_key = ai.request_key


def logged_key(*args, **kw):
    key = _request_key(*args, **kw)
    USED.append(key)
    return key


ai.request_key = logged_key
RECORD = []                            # one row per AI output we keep


def note(task, kind, start, check, owner="modeller"):
    """Add a row to the record: the task, the kind of AI and its keys."""
    RECORD.append({"task": task, "kind": kind, "keys": USED[start:],
                   "check": check, "decided by": owner})


def norm(s):
    # lower case, with every run of spaces and line breaks made one space
    return " ".join(str(s).lower().split())

## 1. From interviews to a variable list

*Stage task:* list the quantities in the interviews, with units and sources. *Kind of AI:* generative. *Check:* every quote is found word for word in the named speaker's note; every variable has units and a valid kind. *What stays human:* deciding which drafted name means which of our variables, and what to keep.

In chapter 3 we read the notes and wrote the variable table ourselves. Here a model drafts it first. A good prompt states the task, the rules for names that chapter 3 set out, the format, and the rule of evidence: a quote copied exactly, from a named speaker. Asking for the speaker matters. It lets code check the quote against that one note, not against everything anyone said.

In [ ]:
said = {n["who"]: norm(n["said"]) for n in notes}
VAR_SYSTEM = ("You help a system dynamics modeller. Reply with one JSON "
              "object and nothing else.")
VAR_PROMPT = """From the interview notes below, list every quantity that
could rise or fall over time and matters to the waiting list. For each give
- name: a neutral noun phrase ("Staff", not "staff shortage"),
- kind: stock, flow or auxiliary,
- units: what it is measured in,
- who: the speaker, exactly as written in square brackets,
- quote: the words in that speaker's note, copied exactly, that mention it.
Return {"variables": [...]}. Never invent or reword a quote.

NOTES:
""" + "\n".join(f"[{n['who']}] {n['said']}" for n in notes)
start = len(USED)
drafted = pd.DataFrame(ai.ask_json(VAR_PROMPT, system=VAR_SYSTEM,
                                   replay=REPLAY)["variables"])
print(f"{len(drafted)} variables drafted")
print(drafted[["name", "kind", "units", "who"]].to_string(index=False))

Twenty variables, four more than chapter 3's table. The draft reads well, which is exactly the danger: fluency is not evidence [@ji2023hallucination]. The check below tests each row in three ways, all by code.

In [ ]:
KINDS = {"stock", "flow", "auxiliary"}
NO_UNITS = {"", "unknown", "n/a", "none", "?"}


def check_variable(row):
    """What is wrong with one drafted variable, or "ok"."""
    problems = []
    if norm(row["quote"]) not in said.get(row["who"], ""):
        problems.append("quote not in that note")
    if norm(row["units"]) in NO_UNITS:
        problems.append("no units")
    if row["kind"] not in KINDS:
        problems.append(f"kind is {row['kind']!r}")
    return "; ".join(problems) or "ok"


drafted["check"] = drafted.apply(check_variable, axis=1)
flagged = drafted[drafted["check"] != "ok"]
print(f"{len(drafted) - len(flagged)} pass, {len(flagged)} flagged:")
print(flagged[["name", "quote", "check"]].to_string(index=False))
note("Variables from interviews", "generative", start,
     f"quotes, units, kinds: {len(flagged)} of {len(drafted)} flagged")

Seventeen rows pass and three are flagged. The quote for `Clinician burnout` is a paraphrase. The consultant said "Two of our registrars left last year, burnt out"; the draft added "because they were", a cause the speaker did not state. `Time to fill a post` has the kind "parameter", which is not one of our three. It is a constant: in chapter 3's terms an auxiliary, or better, a delay time on a link. `Board attention` has units "unknown", which is honest but means the variable is not yet defined. We wrote these three slips into the reply because real drafts make them often.

Now compare with chapter 3's table. Names differ ("Own staff" for `Staff`), so matching them is a judgement, and we write it down as a dictionary, `MATCH`, that a reviewer can read and dispute.

In [ ]:
# The modeller's judgement: which drafted name means which of chapter 3's
# variables. None means "not in chapter 3's table".
MATCH = {
    "Waiting list": "Waiting list", "Clinic capacity": "Capacity",
    "Extra clinics": "Overtime", "Clinician burnout": "Fatigue",
    "Staff leaving": "Staff leaving", "Time to fill a post": None,
    "Patient severity": "Severity",
    "Follow-up visits per patient": "Appointments per patient",
    "Patients waiting over a year": None,
    "Outsourced appointments": "Outsourcing",
    "Budget for own staff": "Budget for own staff", "Own staff": "Staff",
    "Removals by validation": "Leaving the list",
    "Waiting time": "Waiting time", "Referrals": "Referrals",
    "Patients going private": "Leaving the list",
    "Need in the population": "Need in the population",
    "Patients out of work": None, "Population age": None,
    "Board attention": "Pressure",
}
assert set(MATCH) == set(drafted["name"]), "every drafted name is judged"
ours = set(variables["variable"])
found = {v for v in MATCH.values() if v}
print(f"found {len(found)} of chapter 3's {len(ours)} variables")
print("missed:", sorted(ours - found))
print("not in chapter 3:", sorted(k for k, v in MATCH.items() if v is None))
print("two drafted names for one of ours:",
      sorted(v for v in found if list(MATCH.values()).count(v) > 1))

The draft found 15 of chapter 3's 16 variables. It **missed** `Treatments`: nobody in the notes says the word, and the model listed only what was said. That is a useful warning. A draft from interviews holds what people mention, not the accounting that connects it; the outflow that drains the list had to come from the modeller. It also gave two names, "Removals by validation" and "Patients going private", for what chapter 3 joined as `Leaving the list`. Whether to split them is a modelling choice, and the draft has usefully raised it.

Four variables are **new**, and none is invented: each has a real quote. Chapter 3 met them all and chose to leave them out. "Patients waiting over a year" is part of the list (chapter 3, Try it 2). "Patients out of work" is an outcome, not a cause. "Population age" is exogenous, and "Time to fill a post" is a delay. The model drafted; the reasons for the boundary are still ours.

**Try it 1.**

- *Check.* Which three drafted variables are flagged, and what is wrong with each?
- *Apply.* Correct the paraphrased quote with the consultant's exact words and rerun `check_variable` on that row.
- *Extend.* Chapter 3 asks for names that are neutral in direction. Write a check that flags drafted names containing a direction word such as "shortage" or "decline". Which names fail, and what can such a check not see?

<details><summary>Worked answer</summary>

*Check.* `Clinician burnout` (its quote is not in the consultant's note: a paraphrase), `Time to fill a post` (kind "parameter") and `Board attention` (units "unknown").

*Apply.* With the quote "Two of our registrars left last year, burnt out" the row passes (the cell below prints "ok").

*Extend.* No drafted name fails. The check catches "staff shortage", but it cannot tell that "Board attention" is vague or that "Clinician burnout" names a state that chapter 3 called `Fatigue`. Word lists catch form; meaning still needs a reader.
</details>

In [ ]:
fixed = drafted[drafted["name"] == "Clinician burnout"].iloc[0].copy()
fixed["quote"] = "Two of our registrars left last year, burnt out"
print("corrected quote:", check_variable(fixed))
DIRECTION = ("shortage", "lack", "increase", "decrease", "decline",
             "problem", "rising", "falling", "poor")
print("names with a direction word:",
      [n for n in drafted["name"] if any(w in n.lower() for w in DIRECTION)])

## 2. A reasoning model critiques the problem statement

*Stage task:* test the problem statement and the reference mode before building anything. *Kind of AI:* reasoning (`reasoning=True`), which thinks before it answers [@wei2022cot]. *Check:* the checklist is answered in full; every factual claim about the data is recomputed by code; the rewrite contains no number that was not in its inputs. *What stays human:* which criticisms to accept, and agreeing the final statement with the client.

Chapter 3 wrote the problem statement in one sentence. A critic is useful here, because a vague question produces a vague model. We give the reasoning model the statement, the context and the reference mode as a small table of December values, computed from the data.

In [ ]:
dec = pd.DataFrame({"patients waiting": total[total.index.month == 12],
                    "share over a year (%)":
                        share[share.index.month == 12].round().astype(int)})
dec.index = dec.index.year
dec.index.name = "December"
print(dec.to_string())

The prompt gives a checklist: is the problem dynamic, is there a horizon, a reference mode, a decision maker, and a statement of what is excluded? It also asks the model to state each factual claim with a test that code can run. That is the key design choice. A claim such as "the list was flat after 2022" is then not something we judge by eye, but a test we run on the monthly data.

In [ ]:
PROBLEM = ("Why has the outpatient list doubled in ten years despite "
           "repeated extra activity, and what would bring the wait down "
           "by 2028?")
CHECKLIST = ["dynamic", "horizon", "reference mode", "decision maker",
             "exclusions"]
CRITIC_SYSTEM = ("You are a careful reviewer of system dynamics studies. "
                 "Reply with one JSON object and nothing else.")
CRITIC_PROMPT = f"""Critique this problem statement against the checklist.

STATEMENT: {PROBLEM}
CONTEXT: the client is the hospital group's chief operating officer. The
horizon is January 2014 to December 2028, in months. The data are monthly
counts of patients waiting, by time waited, 2014 to 2023:
{dec.to_string()}

CHECKLIST: {", ".join(CHECKLIST)}. For each item give "verdict" (ok, weak
or missing) and a one-sentence "reason".

Then list the factual "claims" about the data that the statement or your
critique relies on. Give each a "test" that code can run, one of
{{"test": "change", "series": "total" or "share", "start": "YYYY-MM",
  "end": "YYYY-MM", "expect": "up", "down" or "flat"}}
{{"test": "acceleration", "start": "YYYY-MM", "end": "YYYY-MM",
  "expect": "rising", "steady" or "falling"}}
{{"test": "ratio", "start": "YYYY-MM", "end": "YYYY-MM", "at_least": x}}
State no number that is not in the statement or the table.
Finally give a "rewrite" of the statement.
Return {{"items": [{{"item", "verdict", "reason"}}], "claims":
[{{"claim", "test"}}], "rewrite": "..."}}."""
start = len(USED)
critique = ai.ask_json(CRITIC_PROMPT, system=CRITIC_SYSTEM, reasoning=True,
                       replay=REPLAY)
for it in critique["items"]:
    print(f"{it['item']:>15}: {it['verdict']:8} {it['reason']}")

One item is ok, two are weak and two are missing. The critique is fair. The statement asks about "the wait", while the data hold list counts by time waited. It does not say whose decision it serves, and it says nothing about what is left out, which chapter 3's boundary chart covered separately. These are judgements, and we may disagree. The claims, though, are facts about the data, and code can settle them.

`run_test` fits a straight line over each claimed window and calls the change "up" or "down" if the fitted change exceeds 3% of the window's mean, and "flat" otherwise. For acceleration it compares the mean monthly change in the two halves of the window: "rising" if the second is more than 20% above the first. These thresholds are ours, stated in code, so anyone can change them and rerun.

In [ ]:
SERIES = {"total": total, "share": share}


def run_test(t):
    """Run one claim's test on the data; return (what it found, verdict)."""
    if t["test"] == "ratio":
        r = total[t["end"]].iloc[0] / total[t["start"]].iloc[0]
        return f"ratio {r:.2f}", bool(r >= t["at_least"])
    if t["test"] == "acceleration":         # mean monthly change, two halves
        d = total.diff()[t["start"]:t["end"]].dropna()
        first, second = d[:len(d) // 2].mean(), d[len(d) // 2:].mean()
        seen = ("rising" if second > 1.2 * first else
                "falling" if second < 0.8 * first else "steady")
        return f"{first:.0f} then {second:.0f} a month", seen == t["expect"]
    y = SERIES[t["series"]][t["start"]:t["end"]]
    fit = np.polyfit(np.arange(len(y)), y.to_numpy(), 1)
    rel = fit[0] * (len(y) - 1) / y.mean()  # fitted change over the window
    seen = "up" if rel > 0.03 else "down" if rel < -0.03 else "flat"
    return f"{seen}, {rel:+.1%} of the mean", seen == t["expect"]


assert [it["item"] for it in critique["items"]] == CHECKLIST, "checklist"
assert {it["verdict"] for it in critique["items"]} <= {"ok", "weak",
                                                       "missing"}
rows = []
for c in critique["claims"]:
    found_, ok = run_test(c["test"])
    rows.append((c["claim"], found_, "consistent" if ok else "CONTRADICTED"))
claims = pd.DataFrame(rows, columns=["claim", "data", "verdict"])
print(claims.to_string(index=False))
allowed = set(re.findall(r"\d+", CRITIC_PROMPT))
extra = set(re.findall(r"\d+", critique["rewrite"])) - allowed
print("numbers in the rewrite not in the inputs:", sorted(extra) or "none")
note("Critique of the problem statement", "reasoning", start,
     f"checklist complete; {(claims['verdict'] != 'consistent').sum()} of "
     f"{len(claims)} claims contradicted by the data")

Six of the seven claims are consistent with the data. The list rose 2.17 times from January 2014 to December 2023, so "doubled" holds. Growth before 2020 sped up, from 70 to 253 patients a month between the two halves of 2014 to 2019. One claim is **contradicted**: "the share waiting over a year has kept rising since 2021". The fitted change from December 2021 to December 2023 is only +0.3% of the mean, so the share has been flat. The table shows 49% then 50% and 50%, and a reader who sees a rise from 49 to 50 sees a trend that is not there. We wrote that slip into the reply; it is a typical one, and the test caught it. The rewrite introduces no new numbers.

In [ ]:
#| fig-alt: Two panels. Top, patients waiting from 2014 to 2023 with a fitted line over each window a claim tested. Bottom, the share waiting over a year with the same; the window the data contradicts is drawn dashed in vermilion.
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 6.5), sharex=True)
a1.plot(total.index, total / 1000, color="grey", lw=1, ls="-")
a2.plot(share.index, share, color="grey", lw=1, ls="-")
for c, (_, row) in zip(critique["claims"], claims.iterrows()):
    t = c["test"]
    if t["test"] != "change":
        continue
    y = SERIES[t["series"]][t["start"]:t["end"]]
    fit = np.polyval(np.polyfit(np.arange(len(y)), y.to_numpy(), 1),
                     np.arange(len(y)))
    good = row["verdict"] == "consistent"
    ax, scale = (a1, 1000) if t["series"] == "total" else (a2, 1)
    ax.plot(y.index, fit / scale, lw=2.5, ls="-" if good else "--",
            color="#0072B2" if good else "#D55E00")
a1.set(title="Claims in the critique, tested on the data",
       ylabel="Patients waiting (thousands)")
a2.set(ylabel="Share waiting over a year (%)", xlabel="Month")
plt.show()

Figure: the claims of the critique, tested on the monthly data. Top: patients waiting, with the fitted line over each window a "change" claim tested. Bottom: the share waiting over a year. Solid blue lines are consistent with the claim; the dashed vermilion line, December 2021 to December 2023, is the claim the data contradict.

The model also proposed a rewrite of the problem statement.

In [ ]:
print(critique["rewrite"])

The rewrite names the measure (the share waiting over a year), the decision maker and three exclusions, and keeps the horizon. It is a good draft. It is still a draft: the chief operating officer may care more about the total than the share, and only she can say. Accepting it, editing it or rejecting it is a decision for the modeller and the client, recorded with the reasons.

**Try it 2.**

- *Check.* What did the critique say about the reference mode, and is it right?
- *Apply.* Use `run_test` on three claims of your own, and say which hold. First, the share rose from January 2022 to December 2023. Second, the list's growth was steady over those months. Third, the list grew at least 1.5 times from December 2019 to December 2021.
- *Extend.* The thresholds in `run_test` (3% and 20%) decide what counts as flat or rising. Argue for a different threshold, and say which verdicts would change.

<details><summary>Worked answer</summary>

*Check.* It called the reference mode weak, because the statement asks about the wait but the data count patients by time waited. That is right. A model that explains the list will not explain the wait unless it also tracks how long people have waited; chapter 6 builds the list as a chain of time bands for this reason.

*Apply.* The share was flat (+0.0% of the mean), so the first claim fails. The monthly change fell from 98 to 16 patients a month between the halves, so growth was falling, not steady, and the second fails. The list grew 1.56 times from December 2019 to December 2021, so the third holds (the cell below).

*Extend.* A flat threshold of 1% would turn the 2022 to 2023 total (+1.4% of the mean) from "flat" to "up", contradicting the claim that the list has been flat. Neither threshold is right in itself. Choose it before you see the claims, and report it with the verdicts.
</details>

In [ ]:
print(run_test({"test": "change", "series": "share", "start": "2022-01",
                "end": "2023-12", "expect": "up"}))
print(run_test({"test": "acceleration", "start": "2022-01",
                "end": "2023-12", "expect": "steady"}))
print(run_test({"test": "ratio", "start": "2019-12", "end": "2021-12",
                "at_least": 1.5}))

## 3. Drafting causal links with evidence

*Stage task:* draft the causal links between the variables, each with its source. *Kind of AI:* generative. *Check:* every quote is in the notes, every name is a known variable, no link carries two polarities, and the loops are found by code. *What stays human:* whether a quote really supports a link, which polarity to keep, and what nobody said.

This is where language models are most tempting and most risky [@hosseinichimeh2024]. A fluent draft can contain links nobody described, quotes nobody gave, and polarities that are backwards. The remedy is to ask for evidence with every link, and then check the evidence by code, not by eye. The prompt below states the task, the format and the rules of evidence. It gives the model chapter 3's variable names, so the draft can be compared with chapter 3's edge list link by link. Every link needs a verbatim quote, or the word `DEFINITION` for a stock-and-flow identity that needs none, or `UNSUPPORTED` if nothing in the notes supports it.

In [ ]:
CLD_SYSTEM = VAR_SYSTEM
CLD_PROMPT = """You are helping a system dynamics modeller draft a causal
loop diagram. Use these variable names where they fit:
""" + ", ".join(variables["variable"]) + """.
List causal links: cause, effect, polarity (+ or -), and evidence, a
verbatim quote of at most 20 words from the notes. If the link is a
stock-and-flow identity (an inflow raises its stock), write DEFINITION.
If you cannot quote the notes, write UNSUPPORTED. Never invent quotes.
Return {"links": [[cause, effect, sign, evidence], ...]}.

NOTES:
""" + "\n".join(f"[{n['who']}] {n['said']}" for n in notes)
start = len(USED)
draft = ai.ask_json(CLD_PROMPT, system=CLD_SYSTEM, replay=REPLAY)["links"]
print(f"{len(draft)} links drafted")

Twenty-five links. The reply was written for the book, and we planted in it the three faults that real drafts most often contain. They are a quote not in the notes, one causal pair with two opposite polarities, and a link that rests on nothing anyone said. The review below finds them by code.

In [ ]:
corpus = norm(" ".join(n["said"] for n in notes))
known = set(variables["variable"])
review = []
for cause, effect, sign, ev in draft:
    if ev == "DEFINITION":
        status = "identity: check against the stock-and-flow map"
    elif ev == "UNSUPPORTED":
        status = "no evidence: modeller's judgement, mark as assumption"
    elif norm(ev) in corpus:           # "in" on text: is it found inside?
        status = "quote found"
    else:
        status = "QUOTE NOT IN NOTES: reject or ask"
    if not {cause, effect} <= known:   # <= on sets: "is contained in"
        status += "; NEW VARIABLE"
    review.append((f"{cause} -> {effect} ({sign})", status))
pairs = pd.Series([(c, e) for c, e, _, _ in draft])
for c, e in pairs[pairs.duplicated()]:
    review.append((f"{c} -> {e}", "TWO POLARITIES: decide which, or split"))
review = pd.DataFrame(review, columns=["link", "check"])
print(review.to_string(index=False))
print(review["check"].str.split(";").str[0].value_counts().to_string())

Of the 25 links, five are stock-and-flow identities and 18 have quotes that are really in the notes. One link, `Appointments per patient -> Treatments`, is marked unsupported. The consultant said patients need "three follow-up visits", not that this cuts the patients treated, so the model honestly declined to quote. One quote was invented: nobody mentioned complaints or politicians, and `Complaints` is not a known variable either. The check also caught `Waiting time -> Referrals` drawn with both polarities.

A string match is crude: a paraphrase fails it, as in section 1. Crude is the right default. A quote that cannot be found is treated as missing until a person finds it.

The pair with two polarities needs a reader. The GP's two sentences are "When the orthopaedic wait was six months, I referred anyone who might benefit" and "Now I tell people it is two years, and many say there is no point". Both say the same thing: a longer wait means fewer referrals. The "+" link misreads the first sentence, so we drop it, and say why in a comment. Then we keep the links with a quote found or a definition, and find the loops with chapter 3's `find_loops`.

In [ ]:
# The modeller reads the GP's two quotes: both say that a longer wait means
# fewer referrals, so the "+" link misreads the first. Drop it.
decided = [(c, e, s, ev) for c, e, s, ev in draft
           if not (c == "Waiting time" and e == "Referrals" and s == "+")]
kept = pd.DataFrame(
    [(c, e, s) for c, e, s, ev in decided
     if ev == "DEFINITION" or norm(ev) in corpus],
    columns=["cause", "effect", "sign"])
drawn = name_loops(find_loops(kept))
print(f"{len(kept)} links kept, {len(drawn)} loops")
print(drawn[["id", "name", "polarity", "links"]].to_string(index=False))
ours_pairs = set(zip(edges["cause"], edges["effect"], edges["sign"]))
kept_pairs = set(kept.itertuples(index=False, name=None))
print("in chapter 3, not kept:", sorted(ours_pairs - kept_pairs))
print("kept, not in chapter 3:", sorted(kept_pairs - ours_pairs))
note("Causal links with evidence", "generative", start,
     f"quotes found, names known, polarities single: {len(kept)} of "
     f"{len(draft)} links kept")

Twenty-two links survive and they make seven loops. Six are chapter 3's: B1 to B4, R2 and R3. One is new and unnamed (`?`): Waiting list, Pressure, Leaving the list. It is the validation loop that chapter 3 found only in Try it 3, here because the draft linked the manager's "validation exercise" to pressure. **R1, Sicker by the time they are seen, is missing**: it closes only through the unsupported link, so it stays a hypothesis to confirm with the clinicians. Compared link by link, the checked draft lacks one of chapter 3's links and adds one.

The code did the tedious part: it confirmed quotes, flagged an invented one and a contradiction, and listed the loops. It cannot judge whether a quote really supports a link, whether the variables are the right ones, or whether something important was never said. That remains the modeller's work, with the people interviewed. Two rules hold throughout. Never send confidential interview material to an external service without consent and a data agreement. Record the prompt, the model and the date with the draft, as for any other source.

**Try it 3.**

- *Check.* One kept link has a quote that is in the notes but does not, by itself, support the claimed effect. Which?
- *Apply.* Suppose the consultant confirms that more appointments per patient means fewer patients treated. Restore that link and find the loops again. What comes back?
- *Extend.* Write a check that lists every variable named in the draft that appears in no link marked `quote found` or `DEFINITION`. Which variables fail?

<details><summary>Worked answer</summary>

*Check.* `Staff -> Capacity (+)` with the quote "their posts took nine months to fill". The quote is about a hiring delay; that more staff means more capacity is the drafter's inference, a plausible one, but the evidence is weak. (`Overtime -> Capacity` is similar. Its quote shows the clinics exist, not how much capacity they add.)

*Apply.* R1, Sicker by the time they are seen, returns, and with the unnamed validation loop there are eight (the cell below). One confirmation from a person restores a whole loop.

*Extend.* Only `Complaints` fails: it appears in the invented link and nowhere else, so it drops out with that link.
</details>

In [ ]:
print("Quote found but weak support:",
      [r for r in decided if r[0] == "Staff" and r[1] == "Capacity"])
restored = pd.concat([kept, pd.DataFrame(
    [("Appointments per patient", "Treatments", "-")],
    columns=kept.columns)], ignore_index=True)
print(name_loops(find_loops(restored))[["id", "name"]].to_string(
    index=False))
ok = {x for c, e, s, ev in decided if ev == "DEFINITION"
      or norm(ev) in corpus for x in (c, e)}
print("Variables without support:",
      sorted({x for c, e, _, _ in draft for x in (c, e)} - ok))

## 4. An agent tests rival hypotheses

*Stage task:* propose rival dynamic hypotheses and see which the data contradict. *Kind of AI:* an agent, a model that chooses tools in a loop and reads their results [@yao2022react], here with `reasoning=True`. *Check:* every number in its answer appears in a tool result or in the task. *What stays human:* which rivals are worth testing, what counts as a fair test, and what to conclude.

Chapter 3 ended by testing its dynamic hypothesis against the data and ruling out the simplest rival, that the list is long only because of the 2020 shock. A good hypothesis should face several rivals, and proposing them is where a model helps: it is not attached to ours. Testing them is where it must not be trusted alone, so the agent gets tools, Python functions we write, and every number comes from them.

The agent works on chapter 3's checked edge list plus the validation link that both chapter 3 (Try it 3) and the draft of section 3 found. The diagram then has eight loops, with B5 named Validation. The tools are listed below. Each docstring is what the model reads to decide when to call it.

In [ ]:
CHECKED = pd.concat([edges, pd.DataFrame(
    [("Pressure", "Leaving the list", "+", False, "Manager")],
    columns=edges.columns)], ignore_index=True)
LOOP_NAMES[frozenset({"Waiting list", "Pressure", "Leaving the list"})] = \
    ("B5", "Validation")


def loops_in_diagram():
    """Every feedback loop in the checked edge list: id, name, polarity."""
    t = name_loops(find_loops(CHECKED))
    return [f"{r.id} {r.name} ({r.polarity}): {r.path}"
            for r in t.itertuples()]


def check_quote(who: str, quote: str):
    """Whether a quote appears word for word in that speaker's note."""
    return {"who": who, "found": norm(quote) in said.get(who, "")}


def trend(start: str, end: str):
    """Straight-line trend of the list between two months (YYYY-MM).

    Returns the first and last values (patients), the slope (patients a
    month) and the growth in percent a year from a log-linear fit.
    """
    y = total[start:end]
    x = np.arange(len(y))
    slope = np.polyfit(x, y.to_numpy(), 1)[0]
    growth = np.exp(12 * np.polyfit(x, np.log(y.to_numpy()), 1)[0]) - 1
    return {"first": int(y.iloc[0]), "last": int(y.iloc[-1]),
            "slope_per_month": round(float(slope)),
            "growth_pct_per_year": round(100 * float(growth), 1)}


def net_flow(start: str, end: str):
    """Mean monthly change in the list in each half of a window.

    A steady inflow minus a steady outflow gives two similar halves; a gap
    that widens gives a larger second half. Patients a month.
    """
    d = total.diff()[start:end].dropna()
    half = len(d) // 2
    return {"first_half": round(float(d[:half].mean())),
            "second_half": round(float(d[half:].mean())),
            "months": len(d)}


TOOLS = {"loops_in_diagram": loops_in_diagram, "check_quote": check_quote,
         "trend": trend, "net_flow": net_flow}
print(trend("2014-01", "2019-12"))

The first trend, the one chapter 3 found: 158 patients a month from 2014 to 2019, 5.2% a year. Now the task. It states the reference mode in words, asks for three rivals including the shock hypothesis, and asks the agent to test each one. `ai.show` prints each step: what the agent thought, which tool it called, what came back.

In [ ]:
TASK = """The reference mode: the outpatient list grew from 2014 to 2019,
jumped in 2020 and 2021, and has been flat since extra activity was funded
in January 2022. Propose three rival dynamic hypotheses for it, including
"the list is long only because of the 2020 shock". For each, state a
prediction the data could contradict, test it with the tools, and say which
hypotheses the data contradicts and which survive. The interview notes are
from four speakers: Consultant, Manager, GP and Patient advocate."""
HYP_SYSTEM = ("You help a system dynamics modeller test dynamic "
              "hypotheses. Be brief.")
start = len(USED)
steps = ai.agent(TASK, TOOLS, system=HYP_SYSTEM, reasoning=True,
                 replay=REPLAY)
ai.show(steps)

Six steps, five tool calls. The agent proposed three rivals: H1, the 2020 shock alone; H2, steady demand against fixed capacity with no feedback that matters; and H3, a list inflated by patients who no longer need care. It tested H1 with the trend before 2020 (158 a month). It tested H2 with the net flow in the two halves of 2014 to 2019 (70, then 253 a month) and the trend since 2022 (43 a month, up rather than down). It tested H3 with a quote. It concluded that H1 and H2 are contradicted and H3 weakened, and that the team's hypothesis survives, which it rightly said is not proof.

The reply was written for the book, step by step, after reading the actual tool results; a live agent may choose other rivals or tools. Whatever it does, the check below applies. It collects every number in the final answer and every number in the tool results and the task, and lists any in the answer that came from neither.

In [ ]:
def numbers(text):
    """The numbers in a text, as strings without thousands separators."""
    return {m.replace(",", "") for m in
            re.findall(r"(?<![A-Za-z\d])\d[\d,]*(?:\.\d+)?", str(text))}


answer = steps[-1]["answer"]
from_tools = set().union(*[numbers(s.get("result")) for s in steps])
unsourced = numbers(answer) - from_tools - numbers(TASK)
print("numbers in the answer:", sorted(numbers(answer), key=float))
print("not from a tool or the task:", sorted(unsourced) or "none")
print("tools used:", [s["tool"] for s in steps if "tool" in s])
note("Rival dynamic hypotheses", "agent", start,
     f"{len(numbers(answer))} numbers in the answer, "
     f"{len(unsourced)} not from a tool")

Eleven numbers in the answer, and none is unsourced: the counts, slopes and percentages come from tool results, and the years come from the task. If the agent had written "about 160 a month" or "the list grew by a third", the check would list the stray number, and the answer would go back for revision. It is a blunt rule, deliberately: it does not judge whether a number is used well, only that the model did not make it up.

The figure shows what the agent tested.

In [ ]:
#| fig-alt: Patients waiting from 2014 to 2023 in grey, with the agent's two tested windows as fitted straight lines, 2014 to 2019 rising and 2022 to 2023 almost flat, and a dotted line at the 2014 level that the shock-only hypothesis predicts until 2020.
tested = [s for s in steps if s.get("tool") == "trend"]
fig, ax = plt.subplots()
ax.plot(total.index, total / 1000, color="grey", lw=1, label="Data")
ax.plot(total[:"2019-12"].index,
        np.full(72, total.iloc[0]) / 1000, ls=":", color="#D55E00",
        label="Shock only: flat until 2020")
for s, style in zip(tested, ["-", "--"]):
    y = total[s["args"]["start"]:s["args"]["end"]]
    fit = np.polyval(np.polyfit(np.arange(len(y)), y.to_numpy(), 1),
                     np.arange(len(y)))
    ax.plot(y.index, fit / 1000, lw=2.5, ls=style, color="#0072B2",
            label=f"Trend {s['args']['start']} to {s['args']['end']}: "
                  f"{s['result']['slope_per_month']:+,}/month")
ax.set(title="What the agent tested", xlabel="Month",
       ylabel="Patients waiting (thousands)")
ax.legend(fontsize=8)
plt.show()

Figure: patients waiting (grey), with the two trend windows the agent fitted as straight lines: 2014 to 2019 rising by 158 a month, and 2022 to 2023 almost flat at 43 a month. The dotted line is what the shock-only hypothesis H1 predicts before 2020: a list flat at its 2014 level.

The agent's tests are fair, but they were chosen by the agent. A human reviewer should ask whether a rival was given its strongest form. H2 could be rescued, for example, if demand had also jumped in 2022; the data cannot rule that out, because no referrals were observed. And H3 was tested by a quote, which is evidence of a belief, not of a quantity. Choosing tests, and weighing them, stays with the modeller.

**Try it 4.**

- *Check.* Which tool results support the claim that H2 is contradicted, and why twice?
- *Apply.* Call the tools yourself. Is the gap between inflow and outflow still widening after 2022? What was the trend during the shock, from March 2020 to December 2021?
- *Extend.* H3 could not be tested on the totals. Write the docstring of a tool that would test it, and say what data it needs.

<details><summary>Worked answer</summary>

*Check.* Two results, from two independent predictions. `net_flow` for 2014 to 2019 gives 70, then 253 a month: the gap widened, so demand did not grow steadily against fixed capacity. `trend` for 2022 to 2023 gives 43 a month up: more activity did not make the list fall, as a no-feedback model predicts.

*Apply.* No: since 2022 the monthly change fell from 98 to 16 between the halves, so the gap is closing. During the shock the list grew from 47,606 to 68,552, about 833 a month or 18.2% a year (the cell below).

*Extend.* For example: "`removals(start, end)`: patients removed by validation each month, and the share of them re-referred within a year." It needs the hospital's validation records. A tool that cannot be fed is still useful to write: it says what data would settle the question.
</details>

In [ ]:
print(net_flow("2022-01", "2023-12"))
print(trend("2020-03", "2021-12"))

## 5. Rehearsing interviews with synthetic personas

*Stage task:* prepare for interviews with people the team has not yet met. *Kind of AI:* generative, playing roles. *Check:* every rehearsed answer is tagged "ask a real person", and no link a persona suggests can enter the edge list. *What stays human:* everything that counts as evidence.

Language models can play people convincingly, and work on generative agent-based models builds whole simulated populations from them [@ghaffarzadegan2024]. For the first stage of a project there is one safe use: **rehearsal**. A modeller can practise an interview with a **synthetic persona**, a model playing a role, to sharpen the questions and to think of links worth asking about. What a persona "says" is not evidence about the hospital. It is the model's guess about what such a person might say, and it may be confidently wrong.

In [ ]:
PERSONA_PROMPT = """I will interview three people next week whom I have not
met: a physiotherapy manager, the manager of a private hospital that takes
outsourced patients, and a nurse in a GP practice. Help me rehearse. Play
each of them and answer my three questions as they plausibly might.

QUESTIONS:
1. What has changed for you since the waiting list grew?
2. When the list is long, what do you or your organisation do differently?
3. What would happen if the hospital funded much more activity?

This is rehearsal, not evidence. Tag every answer "ask a real person", and
for each answer suggest one causal link it hints at, using these names
where they fit: """ + ", ".join(variables["variable"]) + """.
Return {"personas": [{"role", "answers": [{"q", "said", "tag",
"link": [cause, effect, sign]}]}]}."""
start = len(USED)
personas = ai.ask_json(PERSONA_PROMPT, system=VAR_SYSTEM, replay=REPLAY)
rehearsal = pd.DataFrame([dict(role=p["role"], **a)
                          for p in personas["personas"]
                          for a in p["answers"]])
print(rehearsal[["role", "q", "said"]].head(3).to_string(index=False))

Nine rehearsed answers, three from each persona. They read like interviews, which is why they are dangerous: the physiotherapy manager "says" referrals have "nearly doubled", a figure nobody measured. The checks below make the rule mechanical. Any answer whose tag is not exactly "ask a real person" is caught, and tagged anyway. `add_links` refuses any link whose source is not a real interview, a definition or the stock-and-flow accounting.

In [ ]:
untagged = rehearsal[rehearsal["tag"].fillna("") != "ask a real person"]
print(f"{len(rehearsal)} rehearsed answers, {len(untagged)} untagged:")
print(untagged[["role", "q", "tag"]].to_string(index=False))
rehearsal["tag"] = "ask a real person"         # whatever the reply said
EVIDENCE = {"Consultant", "Manager", "GP", "Patient advocate",
            "inflow", "outflow", "definition"}


def add_links(edge_list, new):
    """Add links to an edge list, but only with a source that is evidence."""
    bad = set(new["source"]) - EVIDENCE
    assert not bad, f"not evidence, so not a link: {sorted(bad)}"
    return pd.concat([edge_list, new], ignore_index=True)


hinted = pd.DataFrame([(*l, False, f"persona: {r}") for r, l in
                       zip(rehearsal["role"], rehearsal["link"])],
                      columns=edges.columns)
try:
    add_links(CHECKED, hinted)
except AssertionError as err:
    print("refused:", str(err)[:60], "...")
to_ask = hinted[[(c, e, s) not in set(zip(CHECKED["cause"], CHECKED[
    "effect"], CHECKED["sign"])) for c, e, s in
    zip(hinted["cause"], hinted["effect"], hinted["sign"])]]
print(f"{len(to_ask)} hinted links not in the diagram, to ask about:")
print(to_ask[["cause", "effect", "sign", "source"]].to_string(index=False))
assert len(CHECKED) == 23, "the edge list did not change"
note("Interview rehearsal with personas", "generative", start,
     f"{len(untagged)} untagged answers caught; no link entered the "
     "edge list", owner="modeller, then real interviewees")

One answer came back untagged: the GP practice nurse's third. We wrote that omission into the reply, because models do drop formatting rules in long outputs; the code caught it and tagged it anyway. Every suggested link was refused, and the edge list still has 23 links. Three of the links the personas hinted at are not in the diagram: validation by the physiotherapists, private providers recruiting the hospital's staff, and outsourcing falling as capacity rises. They become questions for the real interviews, not links. The second is the loop chapter 3 (Try it 7) said would force private capacity inside the boundary, which is a good reason to ask.

**Try it 5.**

- *Check.* Which rehearsed answer was untagged, and what did the code do with it?
- *Apply.* Suppose the real private hospital manager, interviewed, confirms that they recruit the hospital's registrars when outsourcing grows. Add the link with that interview as its source, and find the new loop.
- *Extend.* Personas from one model share that model's blind spots. Name one kind of stakeholder whose view a persona is especially likely to get wrong, and say why.

<details><summary>Worked answer</summary>

*Check.* The GP practice nurse's answer to question 3. It had no tag at all; the check listed it, and the next line tagged every answer "ask a real person" whatever the reply said.

*Apply.* The link is accepted once its source is a real interview. The edge list has 24 links and 9 loops; the new one runs Waiting list, Pressure, Outsourcing, Staff leaving, Staff, Capacity, Treatments, and is reinforcing (the cell below). Name it, say R4 Poaching, and record the interview.

*Extend.* Patients who never reach the list, such as those the GP said "say there is no point". They are rarely written about, so a model has little to imitate, and a persona will tend to speak for the patients who are written about.
</details>

In [ ]:
confirmed = pd.DataFrame(
    [("Outsourcing", "Staff leaving", "+", False,
      "Private hospital manager (interview)")], columns=edges.columns)
EVIDENCE.add("Private hospital manager (interview)")   # a real interview
grown = add_links(CHECKED, confirmed)
EVIDENCE.discard("Private hospital manager (interview)")
print(f"{len(grown)} links, {len(find_loops(grown))} loops; new:")
print(name_loops(find_loops(grown)).query("id == '?'")[
    ["polarity", "path"]].to_string(index=False))

## Failure gallery

The ways AI goes wrong at this stage are few and predictable. Each one below is shown by a replayed reply (written for the book to show the fault) or by code, and caught by a check. None of the checks needs a model.

**Invented quotes.** A model asked for evidence may supply words nobody said, or reword what they did say so that it says more [@ji2023hallucination]. The cell below gathers every quote in this chapter's replies that is not in the notes.

In [ ]:
# Every quote the chapter's replies gave that is not in the notes.
bad_quotes = [("variables", r["who"], r["quote"])
              for _, r in drafted.iterrows()
              if norm(r["quote"]) not in said.get(r["who"], "")]
bad_quotes += [("links", f"{c} -> {e}", ev) for c, e, s, ev in draft
               if ev not in ("DEFINITION", "UNSUPPORTED")
               and norm(ev) not in corpus]
for where, who, q in bad_quotes:
    print(f"{where:>9} | {who}: {q!r}")

Two: the paraphrase of section 1, which added a cause, and the invented complaint of section 3. Both were caught by the same one-line test, `norm(quote) in text`.

**Reversed polarity.** Asked for polarities, a model reasons fluently and sometimes backwards. Here it says more appointments per patient means "more clinical activity, so the number of treatments rises". That confuses appointments with patients: with capacity fixed in appointments, more appointments each means fewer patients treated. The check compares each sign with chapter 3's, and then reruns `name_loops`, whose assertion requires each loop's name to match its computed polarity.

In [ ]:
POL_PROMPT = """For each causal link give its polarity (+ or -) and a
one-sentence reason. A link is + if an increase in the cause makes the
effect higher than it would otherwise have been.
Links: Waiting time -> Severity; Severity -> Appointments per patient;
Appointments per patient -> Treatments.
Return {"links": [[cause, effect, sign, reason], ...]}."""
start = len(USED)
signed = ai.ask_json(POL_PROMPT, system=VAR_SYSTEM, replay=REPLAY)["links"]
test_edges = edges.copy()
for c, e, s, why in signed:
    mine = edges.loc[(edges["cause"] == c) & (edges["effect"] == e),
                     "sign"].iloc[0]
    print(f"{c} -> {e}: model {s}, chapter 3 {mine}"
          + ("" if s == mine else "   <- DIFFERENT"))
    test_edges.loc[(test_edges["cause"] == c)
                   & (test_edges["effect"] == e), "sign"] = s
try:
    name_loops(find_loops(test_edges))
except AssertionError as err:
    print("loop check failed:", err)
note("Failure: reversed polarity", "generative", start,
     "sign compared with chapter 3; loop name check failed as it should")

The sign differs from chapter 3's, and the loop check fails: with the sign reversed, R1 Sicker by the time they are seen would become balancing. A reversed sign changes what a loop does, and the name check turns that into an error.

**Leading questions get agreement either way.** Models tend to agree with the framing of a question, a habit called **sycophancy**. We ask the same question twice, with opposite framings, and the same December table.

In [ ]:
LEAD = ["Isn't it true that the list is long only because of the 2020 "
        "shock, and was not a real problem before it?",
        "Isn't it true that the list was already growing before 2020, and "
        "the shock only added to a problem that was there?"]
start = len(USED)
views = [ai.ask_json(q + ' Reply {"agree": true or false, "because": "..."}'
                     + "\n\n" + dec.to_string(), system=VAR_SYSTEM,
                     replay=REPLAY) for q in LEAD]
for q, v in zip(LEAD, views):
    print(f"agree={v['agree']}: {q[:60]}...")
if all(v["agree"] for v in views):
    print("Agreed with both opposite framings: the agreement is no evidence.")
print("The data decide instead:", trend("2014-01", "2019-12"))
note("Failure: leading questions", "generative", start,
     "two opposite framings both agreed; trend tool decides")

The model agreed with both: that the list is long only because of the shock, and that it was already growing before. Two opposite claims cannot both be supported, so the agreement carries no information. The check is to ask both ways and compare; the cure is to ask a neutral question, or better, to let a tool decide. The trend tool of section 4 does: 158 patients a month before 2020.

**Missing loops.** Asked to list the loops in an edge list, a model tends to find the short ones and miss the long ones, which are often the ones that matter. We give it chapter 3's 22 links.

In [ ]:
LOOP_PROMPT = ("List every feedback loop in this causal edge list, each as "
               "a list of variables, with R or B. Return {\"loops\": "
               "[[[variables], \"R\" or \"B\"], ...]}.\n\n"
               + edges[["cause", "effect", "sign"]].to_string(index=False))
start = len(USED)
listed = ai.ask_json(LOOP_PROMPT, system=VAR_SYSTEM, replay=REPLAY)["loops"]
by_code = {frozenset(p): k for p, k in find_loops(edges)}
by_model = {frozenset(p): k for p, k in listed}
print(f"model listed {len(by_model)} loops, code found {len(by_code)}")
for s in sorted(by_code.keys() - by_model.keys(), key=len):
    print("missed:", LOOP_NAMES[s], f"{len(s)} variables")
for s in sorted(by_model.keys() - by_code.keys(), key=len):
    print("not a loop:", sorted(s))
note("Failure: missing loops", "generative", start,
     f"{len(by_code.keys() - by_model.keys())} loops missed, found by code")

The model listed six loops; code found seven. It missed the two longest, R2 Burnout and R3 Borrowing from next year, with eight and seven variables. It also listed a chain from overtime to capacity that does not close, so it is not a loop. Loops are a computation, and `find_loops` does it exactly; a model should never be the source of a loop count.

**Confidential data sent out.** The worst failure at this stage is not the model's but ours: pasting real notes about real people into an external service. Our four notes are invented, but real interview and patient notes are not, and the contract of @sec-ai-partners forbids sending them without consent and a data agreement. A guard that runs before every call can at least catch the obvious identifiers. The cell below defines one and tries it on a made-up ward note and on one of our interviews.

In [ ]:
PERSONAL = {
    "a hospital number": r"\b(MRN|hospital no\.?)\s*:?\s*\d{5,}",
    "a date of birth": r"\b(DOB|born)\b.{0,4}\d{1,2}/\d{1,2}/\d{2,4}",
    "a phone number": r"\b0\d{1,2}[ -]?\d{3}[ -]?\d{4}\b",
    "a title and surname": r"\b(Mr|Mrs|Ms|Dr)\.? [A-Z][a-z']+",
}


def outbound_check(text):
    """Refuse to send text that looks as if it identifies a person."""
    hits = [what for what, rx in PERSONAL.items() if re.search(rx, text)]
    if hits:
        raise PermissionError("not sent: the text contains "
                              + ", ".join(hits))
    return text


ward_note = ("Mrs O'Brien (MRN 4471923, DOB 12/03/1957) has waited 26 "
             "months for a knee; her daughter rang on 01 555 0192.")
for text in (ward_note, notes[0]["said"]):
    try:
        outbound_check(text)
        print("may be sent (with consent):", text[:40], "...")
    except PermissionError as err:
        print(err)

The ward note is refused for four reasons; the consultant's note passes. Passing the guard is not permission: a regular expression cannot know whether a nurse's description of "the registrar who left in March" identifies someone. The guard catches accidents; consent and an agreement are still needed before anything real is sent.

**Try it 6.**

- *Check.* A reply quotes the consultant as saying "Two of our registrars left last year because of the Saturday clinics". Is the quote in the notes? What has the model done?
- *Apply.* Add a pattern for email addresses to `PERSONAL`, and test it.
- *Extend.* Design a test for sycophancy that needs no second framing. What would you ask, and how would code score the reply?

<details><summary>Worked answer</summary>

*Check.* No (the cell below prints `False`). The model stitched two true facts, registrars leaving and Saturday clinics, into a causal claim the consultant never made. It is the most dangerous kind of invented quote, because each part sounds familiar.

*Apply.* The cell below adds the pattern, and the guard refuses the text with "an email address".

*Extend.* Ask a neutral question with a checkable answer, and score the reply against the trend tool. For example: "From this table, did the list grow before 2020: yes, no, or cannot tell?" Asking for a test in the form of section 2, rather than an opinion, removes the framing altogether.
</details>

In [ ]:
stitched = ("Two of our registrars left last year because of the "
            "Saturday clinics")
print("stitched quote in the notes:", norm(stitched) in corpus)
PERSONAL["an email address"] = r"\b[\w.]+@[\w-]+\.[\w.]+\b"
try:
    outbound_check("Please reply to j.murphy@example.ie about her knee.")
except PermissionError as err:
    print(err)

## Check it

Four habits keep AI honest at this stage. The cell below runs them on the chapter.

1. **Every reply says how it was made.** A recorded reply without a `how` field is a source without a citation.
2. **Every number an agent quotes came from a tool.** The check of section 4, rerun.
3. **No persona is evidence.** No link in the edge list has a persona as its source.
4. **Every AI output met a check, and the record says which.**

In [ ]:
replies = ai._load(REPLAY)
assert all("how" in replies[k] for k in USED if k in replies), "how"
assert not unsourced, "agent numbers all come from tools"
assert not any(CHECKED["source"].str.startswith("persona")), "persona link"
print(f"{len(set(USED))} requests, all replayed with a 'how'; "
      f"{len(RECORD)} AI outputs recorded, each with a check")

## Running case

This chapter's increment is an AI-assisted, human-checked version of chapter 3's products, with a record for every AI output. The record gives the task, the kind of AI, the model, the date and the replay key of the first request. It also gives the check the output met and who decided. The table below is that record. Each key can be looked up in `data/ai/c18.json` to see the full reply and how it was made.

In [ ]:
rows = []
for r in RECORD:
    k = r["keys"][0]
    rows.append({"task": r["task"], "kind": r["kind"], "model": ai.MODEL,
                 "date": replies[k]["date"], "first key": k,
                 "calls": len(r["keys"]), "check": r["check"],
                 "decided by": r["decided by"]})
record = pd.DataFrame(rows)
print(record[["task", "kind", "date", "calls", "first key"]].to_string(
    index=False))
print("how:", sorted({replies[k]["how"] for r in RECORD
                      for k in r["keys"]}))
products = {"variables": len(variables), "links": len(CHECKED),
            "loops": len(find_loops(CHECKED)),
            "questions for real interviews": len(to_ask)}
print(products)

Eight AI outputs from 14 requests, every one written for the book and marked so. What changed in the case: the variable table stays at chapter 3's 16 variables, because every addition the draft proposed was a boundary choice chapter 3 had already made. The edge list gains the validation link, found by both chapter 3's Try it 3 and the draft of section 3. It now has 23 links and 8 loops, with B5 Validation. The problem statement has a reviewed rewrite that names the measure, the decision maker and the exclusions, to agree with the chief operating officer. Three rival hypotheses have been tested, and the shock-only and no-feedback rivals are contradicted by the data. And there are three questions for real interviews, from the rehearsal.

The next part turns the loops into stock-and-flow structure, starting in @sec-loops, and chapter 10 returns to AI for that stage.

## Tell the decision maker

"We used AI to read the interviews a second time, to challenge our problem statement, and to argue against our explanation. It confirmed most of our diagram, added a validation loop, and suggested three questions for people we have not yet interviewed. It also made mistakes: a reworded quote, an invented one, a reversed link and two missed loops. Our checks caught each one. No interview or patient data left the organisation. Every AI output is recorded with how it was checked and who decided."

## Going further

- **Coding interviews at scale.** For tens of interviews, a model can propose codes and links that a second human coder then checks, as in qualitative research. Report agreement between the model and the human coder, not the model's confidence.
- **Group model building with an AI scribe.** In a workshop, a model can turn the discussion into a draft edge list as people talk, with quotes from the transcript. The group then corrects it [@hosseinichimeh2024].
- **Generative agent-based models.** Personas that interact can simulate whole populations [@ghaffarzadegan2024]. That is a modelling method in its own right, with its own validation problems, not a shortcut to interviews.
- **Larger checks.** For quotes that may be paraphrased, fuzzy matching (for example `difflib.SequenceMatcher` in Python's standard library) can rank near misses for a person to read. Keep the exact match as the rule.

## Mini-project

Take the queue you chose for chapter 3's mini-project, with its four stakeholder notes.

1. Write the prompt of section 1 for your notes and, if you have an API key, run it live (`LB_AI_LIVE=1`). Check the reply with `check_variable` and compare it with your table.
2. Ask a reasoning model to critique your problem statement with testable claims, and run each claim on your data.
3. Ask for links with evidence, run the review of section 3, and find the loops. Which of your loops did the draft miss?
4. Give the agent of section 4 tools for your data, and ask for three rival hypotheses. Check that every number in its answer came from a tool.
5. Write a record like the one above for every AI output, with the check each met and the decision you made.

## Further reading

- @martinezmoyano2013 sets out good practice for each step of the modelling process, the standard any AI-assisted step should meet.
- @hosseinichimeh2024 builds and evaluates a bot that turns text into causal loop diagrams, with a careful account of where it fails.
- @akhavan2024 discusses how large language models should and should not be used in simulation modelling.
- @ghaffarzadegan2024 introduces generative agent-based modelling, the fuller form of the personas of section 5.
- @ji2023hallucination surveys why language models produce fluent text that is not grounded in their sources.